✅ FAISS + university chunks (RAG)

✅ Your fine-tuned QLoRA adapters

✅ Base meta-llama/Llama-2-7b-hf for fallback

🧠 Chatbot Logic: What Happens Internally
User asks a question

System:

🧠 Embeds the question

🔍 Searches FAISS for relevant chunks

If match found → prepend retrieved context to prompt → pass to QLoRA

If no adapters → fallback to base LLaMA

Print/return response


🧠 Purpose:
This is the RAG + QLoRA only chatbot.

It:

Uses FAISS search over your university text

Uses QLoRA-fine-tuned adapter on meta-llama/Llama-2-7b-hf

Does NOT load DAPT adapter (unsupervised)

No fallback is actually implemented — the comment says fallback, but the code doesn't perform fallback if adapter missing.

Step 1: Imports & Paths
Load FAISS index, chunks, embedding model

In [5]:
import torch
import faiss
import pickle
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from peft import PeftModel
import numpy as np

# ✅ Fixed relative paths for your notebook's location
index_path = "../checkpoints/faiss_embeddings/university_index.faiss"
chunk_path = "../checkpoints/faiss_embeddings/university_chunks.pkl"
adapter_path = "../checkpoints/qlora_finetuned_model/"
model_id = "meta-llama/Llama-2-7b-hf"


 Step 2: Load FAISS Index + Chunks
 Load tokenizer, base model, and apply QLoRA adapter

In [6]:
# Load FAISS index
index = faiss.read_index(index_path)

# Load chunk texts
with open(chunk_path, "rb") as f:
    chunks = pickle.load(f)

# Load embedding model (BGE or MiniLM — use what you indexed with)
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")


Step 3: Load Tokenizer + QLoRA Model
Embed user question, search FAISS, retrieve context

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=True)
tokenizer.pad_token = tokenizer.eos_token

# Load base model in 4-bit (to match QLoRA)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16,
    use_auth_token=True
)

# Load QLoRA adapter (if exists)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Li

Step 4: Define Helper: Search FAISS
Build prompt (instruction + context)

In [8]:
def search_faiss(query, top_k=1):
    embedded = embedding_model.encode([query])
    D, I = index.search(np.array(embedded), top_k)
    return [chunks[i] for i in I[0]]


Step 5: Define Chat Function
Generate answer with QLoRA model

In [9]:
def ask_chatbot(question):
    # Step 1: RAG context
    context = search_faiss(question, top_k=1)[0]

    # Step 2: Compose prompt (Alpaca-style)
    prompt = f"""### Instruction:
{question}

### Context:
{context}

### Response:
"""

    # Step 3: Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Step 4: Extract only the model's response part
    return answer.split("### Response:")[-1].strip()


Step 6: Try It
Output final cleaned response

In [10]:
question = "Where can I book a study room?"
response = ask_chatbot(question)
print("💬 Bot:", response)


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\bitsandbytes\nn\modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


💬 Bot: - [ ] The user is able to view a list of available rooms.
- [ ] The user is able to select a room.
- [ ] The user is able to amend or cancel a booking.
- [ ] The user is able to receive automated booking status confirmations.
- [ ] The user is able to export all personal bookings into an alternative calendar.
- [ ] The user is able to request an individual recurring booking of up to a maximum of five occurrences.
- [ ] The user is able to receive automated email booking status confirmations.
- [ ] The user is able to view all of their bookings in a single view.
- [ ] The user is able to view their personal calendar.
- [ ] The user is able to select multiple rooms at once.
- [ ] The user is able to select a room for a specific time.
- [ ] The user is able to select a room for a specific time.
- [ ] The user is able to select a room for a specific time.
- [ ] The user is able to select a room for a specific time.
- [ ] The user is able to select


🧠 Important Clarification
There is a small mistake in the comment inside the code you posted:

"✅ Base meta-llama/Llama-2-7b-hf for fallback"

BUT:
You don’t implement fallback to base model here.

You directly load the QLoRA adapter.

If it fails (like if file not found), it would crash — no fallback.

There's no conditional try/except or if adapter exists else use base.

⚡ Important:
If you want true fallback (base model answering if no adapter), you would need a try/except logic or model chaining.
